# GapLens walkthrough

This notebook walks through the GapLens pipeline on the crawled data: Cynbit's blog as the client, Proeffico and Canny Technologies as competitors.

It re-runs the analysis stages **in memory** from the cached crawl and embeddings, so it never overwrites `outputs/`. Scores, briefs and evaluation results are read from the files written by `python -m gaplens.pipeline`, so every number here matches `outputs/metrics.json`.

To run it from a fresh clone, first run `python -m gaplens.pipeline --config config.yaml --no-llm` (this crawls the sites, which takes a few minutes).

In [1]:
import json, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
from gaplens.config import load_config
from gaplens.crawl import load_pages
from gaplens import sectionize, cluster

pd.set_option("display.max_colwidth", 90)
cfg = load_config(ROOT / "config.yaml")
P = cfg["paths"]

## 1. Crawled pages

The crawler checks robots.txt with its own User-Agent, waits 1.5 s between requests and caps each site at 60 posts. Automely's robots.txt was unreachable, so the site was skipped, as RFC 9309 requires.

In [2]:
pages = pd.DataFrame(load_pages(P["pages"]))
print(pages.groupby("site").agg(pages=("url", "count"), median_words=("word_count", "median")))
print()
print(pd.read_json(P["crawl_log"], lines=True)[["site", "event", "detail"]].to_string(index=False))

                    pages  median_words
site                                   
Canny Technologies     36         798.0
Cynbit                 25         991.0
Proeffico              39        1601.0

    site               event                                                                                                                                                                                 detail
Automely site_skipped_robots robots.txt unreachable (ConnectTimeout); treated as full disallow per RFC 9309; crawling not permitted for 'GapLensBot/0.1 (content-gap research prototype; polite, robots.txt-aware)'


## 2. Heading-level sections

Each post is split at its H2/H3 headings. Short sections are merged forward, long ones are split, and page furniture (tables of contents, related-post cards, newsletter blocks, paragraphs repeated across a site) is removed.

In [3]:
sc = cfg["sectionize"]
sections = sectionize.sectionize(load_pages(P["pages"]), sc["min_words"], sc["max_words"], sc["boilerplate_min_pages"])
print(len(sections), "sections")
print(sections.groupby("site")["word_count"].describe()[["count", "50%", "min", "max"]])
sections.sample(5, random_state=0)[["site", "heading", "word_count"]]

788 sections
                    count    50%   min    max
site                                         
Canny Technologies  263.0   98.0  60.0  208.0
Cynbit              165.0   84.0  33.0  196.0
Proeffico           360.0  136.0  21.0  350.0


,site,heading,word_count
453,Proeffico,Future of AI in India,77
85,Cynbit,Idea 5: Automated Project Status Reporting,67
545,Canny Technologies,WhatsApp Business API: Complete Guide for Enterprise Automation / Why WhatsApp Busines...,136
312,Proeffico,What is the cost of leaving an ignored operational problem unsolved?,170
334,Proeffico,How Proeffico Approaches Data Modernization Engagements,338


## 3. Embeddings and clustering

Sections are embedded with `BAAI/bge-small-en-v1.5` (cached by the pipeline), reduced with UMAP (cosine, fixed seed) and clustered with HDBSCAN. HDBSCAN marks sections it cannot place confidently as noise (`-1`).

In [4]:
emb = np.load(P["embeddings"])
assert len(emb) == len(sections)
u, h = cfg["umap"], cfg["hdbscan"]
reduced = cluster.reduce(emb, u["n_components"], u["n_neighbors"], u["min_dist"], u["metric"], cfg["project"]["random_seed"])
labels, _ = cluster.cluster(reduced, h["min_cluster_size"], h["min_samples"], h["cluster_selection_method"])
cluster.cluster_metrics(emb, reduced, labels)

/Users/gokulmallabathula/.venvs/gaplens/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/gokulmallabathula/.venvs/gaplens/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


{'n_sections': 788,
 'n_clusters': 50,
 'noise_sections': 126,
 'noise_ratio': 0.1599,
 'cluster_size': {'min': 5, 'median': 10.0, 'max': 81, 'mean': 13.24},
 'silhouette_embedding_cosine': 0.1325,
 'silhouette_umap_euclidean': 0.6012}

The pipeline's saved clustering uses the same seed and parameters. The check below confirms the in-memory run reproduces it.

In [5]:
saved = pd.read_parquet(P["clusters"])
print("identical to saved clusters:", (saved["cluster_id"].to_numpy() == labels).all())

identical to saved clusters: True


## 4. Topic map

Each dot is a section, coloured by site. Sections that sit close together discuss similar topics. The sites occupy largely separate regions: each agency writes about different subjects, in its own style.

In [6]:
import plotly.express as px
tmap = pd.read_parquet(ROOT / "outputs" / "topic_map.parquet")
tmap = tmap[tmap["cluster_id"] >= 0]
fig = px.scatter(tmap, x="x", y="y", color="site", hover_data=["label", "status", "heading"],
                 color_discrete_sequence=["#2a78d6", "#eb6834", "#1baf7a"], height=520)
fig.update_traces(marker=dict(size=6, line=dict(width=1, color="white")))
fig.update_layout(xaxis_visible=False, yaxis_visible=False)
fig

## 5. Coverage, gap status and opportunity score

For each topic cluster GapLens counts sections per site and assigns a status: **Gap** (no Cynbit section), **Weak spot** (Cynbit share below 20% while competitors have at least 3 sections), **Covered** or **Client-only**. Gaps and Weak spots get a transparent opportunity score:

`score = 100 × (0.40·relevance + 0.25·competitor_strength + 0.20·intent + 0.15·(1 − effort))`

In [7]:
gaps = pd.read_csv(ROOT / "outputs" / "gaps.csv")
print(gaps["status"].value_counts().to_string())
gaps.dropna(subset=["score"]).head(10)[["label", "status", "score", "relevance", "competitor_strength", "intent", "effort"]]

status
Gap            30
Weak spot      10
Covered         5
Client-only     5


,label,status,score,relevance,competitor_strength,intent,effort
0,AI business automation benefits,Gap,83.59,1.0000,0.5837,commercial,0.0
1,AI logistics automation,Weak spot,73.92,0.7438,0.6069,commercial,0.0
2,AI operational problem identification,Weak spot,71.82,0.7643,0.4901,commercial,0.0
3,WhatsApp chatbot automation,Weak spot,71.73,0.4433,1.0000,commercial,0.0
4,AI implementation in Indian healthcare,Weak spot,70.33,0.5316,0.5627,transactional,0.0
5,payroll automation ROI,Weak spot,66.89,0.4498,0.7961,commercial,0.0
6,Cloud provider comparison,Weak spot,64.02,0.9213,0.5268,commercial,1.0
7,Custom CRM vs Salesforce cost comparison,Weak spot,59.60,0.7545,0.6169,commercial,1.0
8,Business Process Automation,Gap,54.25,0.6775,0.7658,informational,1.0
9,On-prem LLM deployment for Indian enterprises,Gap,53.23,0.5866,0.6305,commercial,1.0


In [8]:
for r in gaps.dropna(subset=["score"]).head(3).itertuples():
    print(f"{r.label} ({r.score:.1f})\n  {r.reason}\n")

AI business automation benefits (83.6)
  Gap: 17 competitor sections from 1 of 2 competitors, none on Cynbit. High relevance to Cynbit's services (closest service: "AI automation suite: autonomous AI agents and cognitive automation that transform business operations through intelligent execution"). Commercial intent. Effort: expand the existing page https://www.cynbit.in/blog/small-business-ai-automation (similarity 0.86).

AI logistics automation (73.9)
  Weak spot: 21 competitor sections from 1 of 2 competitors, only 1 on Cynbit (5% share). High relevance to Cynbit's services (closest service: "AI automation suite: autonomous AI agents and cognitive automation that transform business operations through intelligent execution"). Commercial intent. Effort: expand the existing page https://www.cynbit.in/blog/document-extraction-ai-benefits (similarity 0.88).

AI operational problem identification (71.8)
  Weak spot: 7 competitor sections from 1 of 2 competitors, only 1 on Cynbit (12% sha

## 6. Content briefs

Briefs are for human writers, never published automatically. The LLM sees only short competitor excerpts and is told not to copy them, not to invent statistics and not to promise rankings. Reference URLs, the internal link and the score are filled in by code.

In [9]:
briefs = json.loads((ROOT / "outputs" / "briefs.json").read_text())
b = briefs[0]
print(b["suggested_title"], f"(score {b['opportunity_score']}, source: {b['source']})\n")
print(b["why_it_matters"], "\n")
for o in b["outline"]:
    print("H2", o["h2"])
    for h3 in o["h3"]:
        print("   H3", h3)

Key Benefits of AI Business Automation for Growing Companies (score 83.59, source: llm)

This topic matters because Cynbit offers an AI automation suite focused on autonomous agents and cognitive automation to transform business operations. It directly addresses commercial intent by showing how AI workflows reduce manual effort and improve decision-making for growing companies. Positioning Cynbit as the implementation partner helps businesses move from manual processes to intelligent execution. 

H2 What AI Business Automation Means for Modern Operations
   H3 Defining autonomous AI agents and cognitive automation
   H3 How intelligent execution differs from manual workflows
H2 Core Benefits of AI Automation in Business
   H3 Reducing manual workload and repetitive tasks
   H3 Improving accuracy and reducing human error
   H3 Scaling operations without proportional staffing increases
H2 AI-Driven Insights and Smarter Decision Making
   H3 Analyzing business performance with AI tools
  

## 7. Evaluation

Cluster quality metrics, the weight sensitivity check and, once hand labels exist, Precision@10/@25 against a baseline that ranks topics by competitor section count only.

In [10]:
m = json.loads((ROOT / "outputs" / "metrics.json").read_text())
print(json.dumps({k: m["clustering"][k] for k in ("n_clusters", "noise_ratio", "silhouette_embedding_cosine", "silhouette_umap_euclidean")}, indent=1))
print("sensitivity (top-10 overlap under ±0.1 weight shifts):", m["sensitivity"]["mean_overlap"], "mean,", m["sensitivity"]["min_overlap"], "min")
p = m["evaluation"]["precision"]
if p:
    for name in ("opportunity_score", "baseline_competitor_count"):
        print(name, {k: v["precision"] for k, v in p[name].items()})

{
 "n_clusters": 50,
 "noise_ratio": 0.1599,
 "silhouette_embedding_cosine": 0.1325,
 "silhouette_umap_euclidean": 0.6012
}
sensitivity (top-10 overlap under ±0.1 weight shifts): 9.88 mean, 9 min
opportunity_score {'p@10': 0.6, 'p@25': 0.6}
baseline_competitor_count {'p@10': 0.8, 'p@25': 0.64}
